# DeepExtractor Glitch Reconstruction Tutorial

#### Hello and welcome to the DeepExtractor tutorial for reconstructing real glitches from LIGO's third observing run (O3)!

### Suppressing unneccesary warnings

In [ ]:
import os
import sys
import warnings

os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/lib/cuda"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/lib/cuda"
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

### Imports

In [ ]:
# Plotting
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])
plt.rcParams['text.usetex'] = False  # Set to True for LaTeX-rendered plots

# Essentials
import pickle
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from gwpy.timeseries import TimeSeries
from gwpy.frequencyseries import FrequencySeries
from pycbc.types import TimeSeries as TimeSeries_pycbc

# DeepExtractor package
from deepextractor.models import UNET2D
from deepextractor.utils.checkpoints import load_torch_model, CHECKPOINT_REAL
from deepextractor.utils.visualization import plot_q_transform
from deepextractor.utils.signal import custom_whiten

# Patch PyCBC TimeSeries with the custom whitening function
TimeSeries_pycbc.custom_whiten = custom_whiten

### Relevant directories

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Assets directory — contains scaler .pkl files
ASSETS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'assets') if os.path.isdir(
    os.path.join(os.path.dirname(os.getcwd()), 'assets')
) else '../../assets'

# Gravity Spy dataset CSVs (update this path to where you stored the CSV files)
PATH_TO_GRAVITY_SPY_DATASETS = 'data/'

### High-confidence Gravity Spy datasets

Let's load in the Gravity Spy datasets so we can pull the relevant glitch data according to their GPS times.

This dataset features glitches that have a Gravity Spy confidence 90%. This means that Gravity Spy is 'confident' that the given class is accurate for the given glitch event.

We can see an example from the O3a dataset below, with features such as GPS time and signal-to-noise ratio (SNR).

In particular, the GPS time will be used to pull the open time series data from GWOSC. This is how we grab the input for DeepExtractor

In [ ]:
gspy_o3a_path = PATH_TO_GRAVITY_SPY_DATASETS+'data_o3a_high_confidence.csv'
gspy_o3b_path = PATH_TO_GRAVITY_SPY_DATASETS+'data_o3b_high_confidence.csv'

data_o3a = pd.read_csv(gspy_o3a_path)
data_o3b = pd.read_csv(gspy_o3b_path)

# Remove duplicates
data_o3a = data_o3a.drop_duplicates(subset=['GPStime'])
data_o3b = data_o3b.drop_duplicates(subset=['GPStime'])

data_o3a.head(1)

In [ ]:
data_o3a.snr.describe()

In [ ]:
data_o3b.snr.describe()

In [ ]:
np.percentile(data_o3a.snr, 90), np.percentile(data_o3b.snr, 90)

In [ ]:
all_data = pd.concat([data_o3a, data_o3b])
np.percentile(all_data.snr, 90)

### Short-Time Fourier Transform (STFT) parameters

These are parameters that are necessary to transform to the time-frequency domain using the STFT. We show in the paper that this is better than doing the mapping in the time-domain.

These parameters result in complex spectrograms of dimension 257x257, which we subsequently decompose into magnitude and phase spectrograms.

We feed the two spectrograms into DeepExtractor's two input channels, which outputs two spectrograms corresponding to the magnitude and phase of the underlying noise ('underneath' the glitch)  

In [ ]:
SAMPLE_RATE = 4096

n_fft = 256*2  # FFT window size
win_length = n_fft//8  # Window length
hop_length = win_length // 2  # Hop length for overlap
window = torch.hann_window(win_length)  # Hann window for smoother STFT

### DeepExtractor Model

Here we define our U-Net architecture and load up the fine-tuned DeepExtractor model weights. 

Our fine-tuned model has been trained on real O3 data, so is capable of reconstructing real glitches.

We also load the scaler needed to standardize the data so DeepExtractor can properly analyze. This is an invertible transform, so we can transform DeepExtractor's output back to the original h(t) space.

In [ ]:
model_name = 'DeepExtractor_257'

# Load real-noise model — downloads from Hugging Face Hub automatically if not found locally.
# This model is paired with scaler.pkl (real O3 LIGO data).
model_fine_tuned = load_torch_model(
    model_name,
    {model_name: UNET2D(in_channels=2, out_channels=2)},
    device=DEVICE,
    checkpoint_filename=CHECKPOINT_REAL,
)
print(f"Model loaded: {model_fine_tuned is not None}")

# Load the real-noise scaler (paired with checkpoint_best_real_noise_base.pth.tar)
scaler_path = os.path.join(ASSETS_DIR, 'scaler.pkl')
with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)

## Glitch reconstruction

The below code grabs glitches (g(t)) according to the observing run (O3a or O3b), the detector (H1 or L1) and the glitch type. It randomly selects glitches from the gravity spy datasets. 

The code stores these glitches, along with the whitened input (h(t)) and SNR, in a dictionary for convenience. The latency is largely due to fetching the open data, you can load a pkl with ready to use examples below this cell.

The workflow is as follows:

1. Get GPS time from dataset.
2. Grab 14s of data where the GPS time is in the centre.
3. Grab 14s of data directly adjacent (before) to the 14s of data that contains the glitch in the middle and calculate the PSD.
4. Use the PSD of the adjacent data to whiten the 14s of data that contains the glitch. This is to ensure that the glitch isn't suppressed during whitening.
5. Grab the middle 2s of the whitened data, the glitch trigger should be in the centre of this 2s.
6. Standardize the data using the scaler we loaded earlier.
7. Calculate spectrograms using the STFT and decomponse into magnitude and phase.
8. Feed through DeepExtractor, which outputs an estimate of the underlying noise in magnitude and phase spectrogram components.
9. Combine DeepExtractors two-channel output (magnitude and phase) into a complex spectrogram.
10. Use the inverse STFT to yield the time-domain noise estimate, which is in the standardized space of our scaler
11. Use the scaler's inverse transform to achieve the underlying noise in its original scaling.
12. Subtract that noise from the time-domain representation of the input (3. on this list).
13. This yields the reconstruction of the glitch.

In [ ]:
NUM_SAMPLES_PER_CLASS = 3 # Let's keep a small dataset for convenience

# Below covers the scope of DeepExtractor; Hanford and Livingston during O3a and O3b
datasets = [data_o3a, data_o3b]
runs = ['O3a', 'O3b']
ifos = ['H1', 'L1']

glitch_labels = ['Blip', 'Tomte', 'Light_Modulation', 'Scattered_Light', 'Whistle'] # We just show a subset of the glitch classes for this example. Uncomment the two lines below it to do reconstructions of all classes
# glitch_labels = list(data_o3a.label.unique())
# glitch_labels.remove('None_of_the_Above')

glitch_dict = {} # The structure will be: Observing Run -> Detector - > Glitch type

for dataset, dataset_name in zip(datasets, runs): # Cycle through the observing runs
    glitch_dict[dataset_name] = {}
    for ifo in ifos: # And now the detectors
        glitch_dict[dataset_name][ifo] = {}
        
        data_filtered = dataset[dataset.ifo == ifo]

        for i, glitch_type in enumerate(glitch_labels): # And now the glitch classes
            
            glitch_dict[dataset_name][ifo][glitch_type] = {'h_input': [], 'g_reconstructed': [], 'snr': []} # For each sample, we have the whitened strain input, the glitch reconstruction, and glitch SNR

            # tqdm.write(f"Processing: Dataset={dataset_name}, IFO={ifo}, Glitch Type={glitch_type}")
 
            data_glitch = data_filtered[data_filtered.label == glitch_type].drop_duplicates(subset=['GPStime']) # Make sure pesky duplicates are removed

            if len(data_glitch) == 0:
                tqdm.write(f"\rNo data for {dataset_name}-{ifo}-{glitch_type}   ", end='', flush=True)
                continue
            
            data_glitch.reset_index(drop=True, inplace=True)
            num_samples = min(NUM_SAMPLES_PER_CLASS, len(data_glitch)) # Just in case some glitch classes have very few samples
            random_indices = np.random.choice(len(data_glitch), size=num_samples, replace=False) # This randomly takes samples from the Gravity Spy dataset. Feel free to select specific ones

            progress_bar = tqdm(total=num_samples, desc=f"{dataset_name}-{ifo}-{glitch_type}", position=0, leave=False, dynamic_ncols=True)

            glitch_time_series = []
            
            MAX_RETRIES = 3  # Number of times to retry fetching data

            for j, idx in enumerate(random_indices):
                retry_count = 0
                while retry_count < MAX_RETRIES:
                    snr = data_glitch.snr.iloc[idx+retry_count]
                    gps_time = data_glitch.GPStime.iloc[idx+retry_count]
                    try:
                        segment_for_psd = TimeSeries.fetch_open_data(
                            ifo, gps_time - 7 - 14, gps_time - 7, sample_rate=SAMPLE_RATE
                        )
                        glitch = TimeSeries.fetch_open_data(
                            ifo, gps_time - 7, gps_time + 7, sample_rate=SAMPLE_RATE
                        )
                        break  # If successful, exit the loop
                    except Exception as e:
                        retry_count += 1
                        print(f"Retry {retry_count}/{MAX_RETRIES} - Error fetching GPS time {gps_time} for IFO {ifo}: {e}")
                        if retry_count == MAX_RETRIES:
                            print(f"Skipping GPS time {gps_time} for IFO {ifo} after {MAX_RETRIES} failed attempts.")
                            continue

                segment_for_psd = np.asarray(segment_for_psd) 
                glitch = np.asarray(glitch)
                
                segment_for_psd_pycbc = TimeSeries_pycbc(segment_for_psd, delta_t=1. / SAMPLE_RATE) #PyCBC for whitening 
                glitch_pycbc = TimeSeries_pycbc(glitch, delta_t=1. / SAMPLE_RATE)
                
                _, psd = segment_for_psd_pycbc.whiten(2,1, remove_corrupted=False, return_psd=True) # Calculate PSD with the data adjacent to the glitch
                white_glitch, _ = glitch_pycbc.custom_whiten(psd, return_psd=True) # Use that PSD to whiten the glitch
                
                if np.any(np.isnan(np.array(white_glitch))):
                    print(f"Skipping GPS time {gps_time} for IFO {ifo} due to NaN values in the whitened data.")
                    continue
                
                white_glitch = np.asarray(white_glitch)
                white_glitch_centered = white_glitch[len(white_glitch) // 2 - 4096:len(white_glitch) // 2 + 4096]
                white_glitch_centered_scaled = scaler.transform(white_glitch_centered.reshape(-1, 1)).reshape(white_glitch_centered.shape) # Standardize, required for proper reconstructions
                
                # Convert to PyTorch tensor
                glitch_tensor = torch.tensor(white_glitch_centered_scaled, dtype=torch.float32)
                
                # Apply STFT
                stft_result = torch.stft(
                    glitch_tensor, 
                    n_fft=n_fft, 
                    hop_length=hop_length, 
                    win_length=win_length, 
                    window=window, 
                    return_complex=True
                )
                
                # Extract magnitude and phase
                magnitude = torch.abs(stft_result)
                phase = torch.angle(stft_result)
                
                # Stack magnitude and phase into a 2-channel tensor
                stft_mag_phase = torch.stack([magnitude, phase], dim=0)
                stft_mag_phase = stft_mag_phase.unsqueeze(0)
                
                # Convert scaled data to PyTorch tensors
                h_stft = stft_mag_phase.float().to(DEVICE)
                
                # Model prediction
                with torch.no_grad():
                    n_hat_stft_fine_tuned = model_fine_tuned(h_stft)
                
                n_hat_stft_fine_tuned = n_hat_stft_fine_tuned.cpu() # Noise estimate in STFT
                
                # Separate magnitude and phase
                magnitude_fine_tuned = n_hat_stft_fine_tuned[:, 0, :, :]  # First channel is the magnitude
                phase_fine_tuned = n_hat_stft_fine_tuned[:, 1, :, :]

                # Convert to complex spectrogram
                real_part_fine_tuned = magnitude_fine_tuned * torch.cos(phase_fine_tuned)
                imag_part_fine_tuned = magnitude_fine_tuned * torch.sin(phase_fine_tuned)
                stft_complex_fine_tuned = torch.complex(real_part_fine_tuned, imag_part_fine_tuned)
                
                # Perform the iSTFT to get the standardized time series
                n_hat_t_scaled_fine_tuned = torch.istft(
                    stft_complex_fine_tuned,
                    n_fft=n_fft,
                    hop_length=hop_length,
                    win_length=win_length,
                    window=window,
                )
                n_hat_t_scaled_fine_tuned = n_hat_t_scaled_fine_tuned.numpy().squeeze()
                
                # Scale back the inverse transformed data
                n_hat_t_fine_tuned = scaler.inverse_transform(
                    n_hat_t_scaled_fine_tuned.reshape(-1, n_hat_t_scaled_fine_tuned.shape[-1])
                ).reshape(n_hat_t_scaled_fine_tuned.shape)
                
                 # Reconstruct the glitch by subtracting the predicted noise
                g_hat_fine_tuned = white_glitch_centered - n_hat_t_fine_tuned

                # Store in the dictionary
                glitch_dict[dataset_name][ifo][glitch_type]['h_input'].append(white_glitch) # We save all 10s of data so that we can plot decent Q-scans for middle 2s
                glitch_dict[dataset_name][ifo][glitch_type]['g_reconstructed'].append(g_hat_fine_tuned)
                glitch_dict[dataset_name][ifo][glitch_type]['snr'].append(snr)

                # Update progress bar
                progress_bar.update(1)
                progress_bar.set_description(f"{dataset_name}-{ifo}-{glitch_type} ({j+1}/{num_samples})", refresh=True)

            progress_bar.close()

### Saving the glitches

If you want to save this glitch dictionary, uncomment the below

In [ ]:
# with open(DATA_DIR+'glitch_dict.pkl', 'wb') as f:
#     pickle.dump(glitch_dict, f)

### Plotting the glitches

The below code plots examples from the glitch dictionary created above. As an example, we just look at a subset of classes from H1 during O3a and plot 3 samples per class.

Rows are individual examples, columns are glitch types. 

Uncomment the code directly below to load ready-to-use examples of glitch reconstructions so you don't have the load the above ones.

In [ ]:
# with open(os.path.join(ASSETS_DIR, 'glitch_dict_example.pkl'), 'rb') as f:
#     glitch_dict = pickle.load(f)

In [ ]:
NUM_SAMPLES_TO_PLOT = 3

# Select dataset and IFO
dataset_name = 'O3a'
ifo = 'H1'

glitch_types = ['Blip', 'Tomte', 'Light_Modulation', 'Scattered_Light', 'Whistle']

# Create subplots: 2 rows (examples) x 5 cols (glitch types)
fig, axes = plt.subplots(nrows=NUM_SAMPLES_TO_PLOT, ncols=len(glitch_types), figsize=(20, 8))

# Time axis (assuming the same length for all examples)
t = np.linspace(-1, 1, 8192)  # Adjust based on actual length

for col, glitch_type in enumerate(glitch_types):
    h_t = glitch_dict[dataset_name][ifo][glitch_type]['h_input']
    h_t = np.array(h_t)[:, len(h_t[0])//2-SAMPLE_RATE:len(h_t[0])//2+SAMPLE_RATE] # We only want the middle 2s around the glitch
    g_reconstructed = glitch_dict[dataset_name][ifo][glitch_type]['g_reconstructed']
    snr_values = glitch_dict[dataset_name][ifo][glitch_type]['snr']  # Get SNR values
    
    for row in range(NUM_SAMPLES_TO_PLOT):  # Three examples per glitch type
        ax = axes[row, col]
        
        if row < len(g_reconstructed):  # Ensure there is data
            ax.plot(t, h_t[row], c='gray', alpha=0.4, label="h_input")
            ax.plot(t, g_reconstructed[row], c='C0', label="g_reconstructed")

            # Set individual axis limits
            ax.set_xlim(t[0], t[-1])
            ax.set_ylim(min(h_t[row].min(), g_reconstructed[row].min()), 
                        max(h_t[row].max(), g_reconstructed[row].max()))
            
            # Add SNR as a subtitle above each plot
            formatted_glitch_type = glitch_type.replace("_", " ")  # Replace underscores with spaces
            ax.set_title(r"$\mathit{" + formatted_glitch_type + r"}$" + f"\nSNR: {snr_values[row]:.2f}" if row == 0 else f"SNR: {snr_values[row]:.2f}", fontsize=12)


        # Axis labels
        if col == 0:
            ax.set_ylabel("Amplitude", fontsize=14)
        if row == NUM_SAMPLES_TO_PLOT - 1:
            ax.set_xlabel("Time (s)", fontsize=14)
        
        ax.grid(True)

plt.tight_layout()
plt.show()


### Glitch mitigation

Let's use the glitch reconstructions for glitch mitigation. We will take the first sample for each class and plot Q-scans before and after subtracting the reconstruction.

We will again show the corresponding time series below for reference

In [ ]:
# Select dataset and IFO
dataset_name = 'O3a'
ifo = 'H1'

glitch_types = ['Blip', 'Tomte', 'Light_Modulation', 'Scattered_Light', 'Whistle']

# Create subplots: 3 rows (Q-transform of h_t, Q-transform of n_hat_t, time series) x glitch types columns
fig, axes = plt.subplots(nrows=3, ncols=len(glitch_types), figsize=(20, 10))

# Time axis (assuming 8192 samples, adjust as needed)
t = np.linspace(-1, 1, 8192)

for col, glitch_type in enumerate(glitch_types):
    h_t = np.array(glitch_dict[dataset_name][ifo][glitch_type]['h_input'][0])  # Original 14s strain data
    h_t_centered = h_t[len(h_t)//2-SAMPLE_RATE:len(h_t)//2+SAMPLE_RATE] # The middle 2s for the time series plot
    g_reconstructed = (glitch_dict[dataset_name][ifo][glitch_type]['g_reconstructed'][0])  # Reconstructed glitch
    n_hat_t = h_t.copy()
    n_hat_t[len(n_hat_t)//2-SAMPLE_RATE:len(n_hat_t)//2+SAMPLE_RATE] -= g_reconstructed  # Predicted background noise, after subtraction

    T = len(n_hat_t) / SAMPLE_RATE
    t_inj = T / 2 # This is just for cropping the Q-scans

    snr_values = glitch_dict[dataset_name][ifo][glitch_type]['snr']  # SNR values

    # Row 1: Q-transform of h_t
    ax1 = axes[0, col]
    plot_q_transform(h_t, crop = (t_inj, 2), ax=ax1, colourbar=False)  # Assuming plot_q_transform accepts an axis argument
    ax1.set_title(f"{glitch_type}", fontsize=14)  # Glitch type title

    # Row 2: Q-transform of n_hat_t
    ax2 = axes[1, col]
    plot_q_transform(n_hat_t, crop = (t_inj, 2), ax=ax2, colourbar=False)
    ax2.set_title(f"SNR: {snr_values[0]:.2f}", fontsize=12)  # SNR subtitle

    # Row 3: Time series plot of h_t and g_reconstructed
    ax3 = axes[2, col]
    ax3.plot(t, h_t_centered, c='gray', alpha=0.4, label="Input")  
    ax3.plot(t, g_reconstructed, c='C0', label="Reconstructed glitch")

    # Formatting
    ax3.set_xlim(t[0], t[-1])
    ax3.set_xlabel("Time (s)", fontsize=12)
    ax3.set_ylabel("Amplitude", fontsize=12 if col == 0 else 10)
    ax3.legend()
    ax3.grid(True)

plt.tight_layout()
plt.show()


### Thank you for your interest in DeepExtractor. Please contact me on chat.ligo or by emailing me at tom.dooney@ou.nl if you have any questions or feedback.

### DeepExtractor is still in early stages of development, and investigation into its performance is ongoing. Your feedback is greatly appreciated!

### Have a nice glitch reconstruction!